In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_54_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 55 ###

### cell 55 optimized ###

# Define question strings (unchanged)
question_of_interest_cell67 = \
    'Which of the following cloud computing platforms do you use? (Select all that apply)'
question_of_interest_old_cell67 = \
    'Which of the following cloud computing platforms do you use on a regular basis?'
question_of_interest_old_2 = \
    'Which of the following cloud computing services have you used at work or school in the last 5 years?'

# 1) Clean up the 2018 platform column names in one GPU regex replace
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
      .str.replace(
          r'(Amazon Web Services \(AWS\)|Google Cloud Platform \(GCP\)|Microsoft Azure)(?! )',
          r'\1 ',
          regex=True
      )
)

# 2) Standardize question text across DataFrames
for df in (responses_df_2021, responses_df_2020, responses_df_2019_cell10, responses_df_2018_cell10):
    df.columns = (
        df.columns
          .str.replace(question_of_interest_old_cell67, question_of_interest_cell67, regex=False)
          .str.replace(question_of_interest_old_2, question_of_interest_cell67, regex=False)
    )

# 3) GPU-friendly subsetting, renaming, dropping and year-tagging
def grab_subset_of_data_67(df, question, year):
    # select only the columns containing the question text
    subset = df.filter(like=question)
    # strip off the question prefix (up to the last hyphen) and any extra whitespace
    cleaned = subset.columns.str.replace(r'^.*?-\s*', '', regex=True).str.strip()
    subset.columns = cleaned
    # drop rows where all platform columns are null (GPU)
    subset = subset.dropna(how='all')
    # add the year column (GPU)
    subset['year'] = year
    return subset

# 4) Build a list of per-year DataFrames and concat on GPU
dfs_map = {
    '2018': responses_df_2018_cell10,
    '2019': responses_df_2019_cell10,
    '2020': responses_df_2020,
    '2021': responses_df_2021,
    '2022': responses_df_2022_cell10
}
cloud_dfs = [
    grab_subset_of_data_67(df, question_of_interest_cell67, yr)
    for yr, df in sorted(dfs_map.items())
]
cloud_df_combined = pd.concat(cloud_dfs, ignore_index=True)

# 5) Count non-null selections per platform per year (GPU boolean→sum)
cloud_df_combined_counts = (
    cloud_df_combined
      .notnull()
      .groupby('year')
      .sum()
      .reset_index()
)

# 6) Compute year totals and broadcast division for percentages on GPU
year_totals = (
    cloud_df_combined['year']
      .value_counts(sort=False)
      .sort_index()
)
cloud_df_combined_percentages = cloud_df_combined_counts.copy()
for col in cloud_df_combined_counts.columns.drop('year'):
    cloud_df_combined_percentages[col] = (
        cloud_df_combined_counts[col] / year_totals * 100
    )

# 7) Melt only the three platform columns (now without trailing spaces) on GPU
to_melt = [
    'Amazon Web Services (AWS)',
    'Google Cloud Platform (GCP)',
    'Microsoft Azure'
]
df_cell67 = cloud_df_combined_percentages.melt(
    id_vars=['year'],
    value_vars=to_melt,
    var_name='',      # platform column has empty name
    value_name='value'
)

df_cell67.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_q5/checkpoints/post_cell_55_try_2.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_55_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[55], f)


In [ ]:
opt_output = Out.get(4)